# Module 09 - Transformer Block

Use this notebook after implementing the Module 09 library scaffolds. The goal is to make the transformer block physical: LayerNorm normalizes each token vector, the FFN works independently per position, residuals preserve the stream, and `TransformerLM` turns a stack of blocks into `(B, T, V)` logits.

1. Read the lesson page (`docs/modules/09-transformer-block.md`).
2. Open this notebook with `./notebook.sh 09`.
3. Answer the `Question:` / `Answer:` cells below.
4. When you're ready, ask a coding agent to grade your notebook.

Partial work is fine. Blank `Answer: ""` strings are skipped, not counted wrong. If you'd like a hint instead of a grade, write the request inline in the answer string and the agent will tutor first.

In [ ]:
from __future__ import annotations

import math
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

import g2c
from g2c.notebook_extras.transformer import make_lm_with_block, train_tiny_transformer
from g2c.transformer import Block, FeedForward, LayerNorm, TransformerLM

_ = torch.manual_seed(0)
repo_root = Path(g2c.__file__).resolve().parents[1]
print("MPS available:", torch.backends.mps.is_available())

## Before the Notebook

Implement `LayerNorm.forward`, `FeedForward.forward`, `Block.forward`, and `TransformerLM.forward` first. The notebook calls those methods directly, so use the tests as the implementation guide before treating these cells as experiments.

In [ ]:
"Run from the terminal: .venv/bin/python -m pytest tests/test_transformer.py -x"
"Question: Which transformer test is the next one failing, and what exact contract does it point at?"
"Answer: "

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_transformer.py"],
    cwd=repo_root,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

assert result.returncode == 0, "Module 09 transformer tests are not passing yet."
print("Module 09 transformer tests passed.")

## Exercise 1 - LayerNorm on One Batch

Predict the output statistics before running the check. LayerNorm should normalize across the last dimension only, so each `(batch, position)` slice gets its own mean and variance.

In [ ]:
"Question: For an input shaped (2, 3, 8), how many separate means does LayerNorm compute?"
"Answer: "
"Question: Why does LayerNorm use unbiased=False for the variance?"
"Answer: "

In [ ]:
ln = LayerNorm(embedding_dim=8)
x = torch.randn(2, 3, 8) * 3.0 + 7.0
y = ln(x)

print("input shape:", tuple(x.shape))
print("output shape:", tuple(y.shape))
print("per-position means after LN:\n", y.mean(dim=-1))
print("per-position variances after LN:\n", y.var(dim=-1, unbiased=False))

assert y.shape == x.shape
assert torch.allclose(y.mean(dim=-1), torch.zeros(2, 3), atol=1e-5)
assert torch.allclose(y.var(dim=-1, unbiased=False), torch.ones(2, 3), atol=1e-4)

## Exercise 2 - The FFN Is Per-Position

Attention mixes information across positions. The FFN does not. It applies the same two-layer MLP to every position independently.

In [ ]:
"Question: If you mutate token position 2, which FFN outputs should change?"
"Answer: "
"Question: Why does the FFN expand to hidden_dim = 4 * embedding_dim before projecting back down?"
"Answer: "

In [ ]:
torch.manual_seed(0)
ffn = FeedForward(embedding_dim=8, hidden_dim=32)
x1 = torch.randn(1, 4, 8)
x2 = x1.clone()
x2[0, 2, :] = torch.randn(8)

y1 = ffn(x1)
y2 = ffn(x2)

for pos in range(4):
    changed = not torch.allclose(y1[0, pos], y2[0, pos], atol=1e-6)
    print(f"position {pos}: changed={changed}")

assert torch.allclose(y1[0, 0], y2[0, 0], atol=1e-6)
assert torch.allclose(y1[0, 1], y2[0, 1], atol=1e-6)
assert not torch.allclose(y1[0, 2], y2[0, 2], atol=1e-4)
assert torch.allclose(y1[0, 3], y2[0, 3], atol=1e-6)

## Exercise 3 - Residual Identity Check

A transformer block should pass the residual stream through unchanged when both sublayers write zero updates. This is the concrete test for the residual structure.

In [ ]:
"Question: Why does zeroing the final attention and FFN projections make each sublayer output zero?"
"Answer: "
"Question: What would this cell print if Block.forward forgot the residual additions?"
"Answer: "

In [ ]:
torch.manual_seed(0)
block = Block(embedding_dim=8, num_heads=2, hidden_dim=16, causal=True)
with torch.no_grad():
    block.attn.out_proj.W.zero_()
    block.attn.out_proj.b.zero_()
    block.ffn.fc2.W.zero_()
    block.ffn.fc2.b.zero_()

x = torch.randn(2, 4, 8)
y = block(x)
print("max absolute difference from identity:", (y - x).abs().max().item())
assert torch.allclose(y, x, atol=1e-6)

## Exercise 4 - Pre-Norm vs Post-Norm

The production block is pre-norm: `x = x + sublayer(LN(x))`. This experiment defines a post-norm variant and trains both on the same tiny next-token task. The task is intentionally artificial: a repeated integer pattern, not Shakespeare or real text. That keeps the data simple so the curves mostly reflect block design rather than tokenizer or corpus effects.

In [ ]:
class PostNormBlock(Block):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.ln1(x + self.attn(x))
        x = self.ln2(x + self.ffn(x))
        return x

### The Toy Training Stream

We train on one repeated token pattern. The model sees random context windows from this flat stream and predicts the same stream shifted one token to the right. The model vocabulary is 50, but the stream only uses token IDs 0 through 9, so `log(50)` is the uniform-random baseline for the model's output space.

The important point is not language quality. The important point is whether the transformer block can learn a simple repeated next-token structure while gradients flow through several stacked blocks.

In [ ]:
pattern = torch.tensor(
    [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 3, 4, 5, 6, 7, 0],
    dtype=torch.long,
)
toy_ids = pattern.repeat(200)
toy_vocab_size = 50
toy_context_length = 32

example_x = toy_ids[:toy_context_length]
example_y = toy_ids[1 : toy_context_length + 1]

print("pattern:", pattern.tolist())
print("pattern length:", len(pattern))
print("training stream tokens:", len(toy_ids))
print("unique tokens in stream:", sorted(toy_ids.unique().tolist()))
print("model vocab size:", toy_vocab_size)
print("example x[:16]:", example_x[:16].tolist())
print("example y[:16]:", example_y[:16].tolist())
print("y is x shifted one token to the right")

In [ ]:
experiment_steps = 300
torch.manual_seed(1)
pre_norm_model = make_lm_with_block(
    Block,
    vocab_size=toy_vocab_size,
    max_seq_len=toy_context_length,
)
pre_curve = train_tiny_transformer(
    pre_norm_model,
    toy_ids,
    steps=experiment_steps,
    lr=1e-3,
    context_length=toy_context_length,
)

torch.manual_seed(1)
post_norm_model = make_lm_with_block(
    PostNormBlock,
    vocab_size=toy_vocab_size,
    max_seq_len=toy_context_length,
)
post_curve = train_tiny_transformer(
    post_norm_model,
    toy_ids,
    steps=experiment_steps,
    lr=1e-3,
    context_length=toy_context_length,
)

plt.figure(figsize=(8, 4))
plt.plot([s for s, _ in pre_curve], [v for _, v in pre_curve], label="pre-norm")
plt.plot([s for s, _ in post_curve], [v for _, v in post_curve], label="post-norm")
plt.axhline(math.log(toy_vocab_size), color="gray", linestyle="--", label="uniform baseline")
plt.xlabel("step")
plt.ylabel("cross entropy")
plt.title("Pre-norm vs post-norm")
plt.legend()
plt.show()

In [ ]:
"Question: Which curve is smoother in your run, and what does that suggest about the residual gradient path?"
"Answer: "
"Question: If post-norm trains fine after adding warmup in Module 10, does that make pre-norm unnecessary? Why or why not?"
"Answer: "

## Exercise 5 - Strip Residuals or Strip LayerNorm

These variants are intentionally bad. The point is to see that residuals and normalization are not cosmetic additions around attention; they are what make stacked blocks trainable.

In [ ]:
class ResidualFreeBlock(Block):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.attn(self.ln1(x))
        x = self.ffn(self.ln2(x))
        return x


class NoNormBlock(Block):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(x)
        x = x + self.ffn(x)
        return x


def run_depth_sweep(block_cls, depths: list[int], *, steps: int = 200) -> dict[int, list[tuple[int, float]]]:
    curves: dict[int, list[tuple[int, float]]] = {}
    for depth in depths:
        torch.manual_seed(2)
        model = make_lm_with_block(
            block_cls,
            num_layers=depth,
            vocab_size=toy_vocab_size,
            embedding_dim=48,
            num_heads=4,
            max_seq_len=toy_context_length,
        )
        curves[depth] = train_tiny_transformer(
            model,
            toy_ids,
            steps=steps,
            lr=1e-3,
            batch_size=32,
            context_length=toy_context_length,
        )
    return curves

In [ ]:
depths = [1, 2, 4, 8]
residual_free_curves = run_depth_sweep(ResidualFreeBlock, depths, steps=200)

plt.figure(figsize=(8, 4))
for depth, curve in residual_free_curves.items():
    plt.plot([s for s, _ in curve], [v for _, v in curve], label=f"depth {depth}")
plt.axhline(math.log(toy_vocab_size), color="gray", linestyle="--", label="uniform baseline")
plt.xlabel("step")
plt.ylabel("cross entropy")
plt.title("Residual-free blocks")
plt.legend()
plt.show()

In [ ]:
"Question: At what depth did the residual-free model stop making meaningful progress?"
"Answer: "
"Question: How does this connect to the residual-stream explanation in the module?"
"Answer: "

In [ ]:
no_norm_curves = run_depth_sweep(NoNormBlock, depths, steps=200)

plt.figure(figsize=(8, 4))
for depth, curve in no_norm_curves.items():
    plt.plot([s for s, _ in curve], [v for _, v in curve], label=f"depth {depth}")
plt.axhline(math.log(toy_vocab_size), color="gray", linestyle="--", label="uniform baseline")
plt.xlabel("step")
plt.ylabel("cross entropy")
plt.title("No-LayerNorm blocks")
plt.legend()
plt.show()

In [ ]:
"Question: Does removing LayerNorm fail in the same way as removing residuals, or is the curve different?"
"Answer: "
"Question: Which symptom would make you suspect scale instability rather than missing gradient flow?"
"Answer: "

## Exercise 6 - TransformerLM Shape and Parameter Budget

The full language model returns one vocab-sized logit vector per position. Then parameter counting tells you where the model's capacity lives.

In [ ]:
torch.manual_seed(0)
m = TransformerLM(vocab_size=32, embedding_dim=32, num_layers=2, num_heads=4, max_seq_len=16)
ids = torch.randint(0, 32, (3, 10))
logits = m(ids)
print("ids:", tuple(ids.shape))
print("logits:", tuple(logits.shape))
assert logits.shape == (3, 10, 32)

In [ ]:
def count_params(model) -> int:
    return sum(p.numel() for p in model.parameters())


def analytical_transformer_params(
    *,
    vocab_size: int,
    embedding_dim: int,
    num_layers: int,
    max_seq_len: int,
    hidden_dim: int | None = None,
) -> int:
    D = embedding_dim
    H_ff = 4 * D if hidden_dim is None else hidden_dim
    token_embed = vocab_size * D       # also serves as the unembedding (tied)
    pos_embed = max_seq_len * D
    mha = 4 * (D * D + D)
    ffn = (D * H_ff + H_ff) + (H_ff * D + D)
    lns_per_block = 4 * D
    per_block = mha + ffn + lns_per_block
    final_ln = 2 * D
    head_bias = vocab_size              # only the per-token bias; the matrix is tied
    return token_embed + pos_embed + num_layers * per_block + final_ln + head_bias


configs = [
    dict(vocab_size=100, embedding_dim=32, num_layers=2, num_heads=4, max_seq_len=32),
    dict(vocab_size=100, embedding_dim=64, num_layers=4, num_heads=4, max_seq_len=32),
    dict(vocab_size=500, embedding_dim=64, num_layers=2, num_heads=4, max_seq_len=64),
]

for cfg in configs:
    model = TransformerLM(**cfg)
    actual = count_params(model)
    expected = analytical_transformer_params(
        vocab_size=cfg["vocab_size"],
        embedding_dim=cfg["embedding_dim"],
        num_layers=cfg["num_layers"],
        max_seq_len=cfg["max_seq_len"],
    )
    untied_total = expected + cfg["vocab_size"] * cfg["embedding_dim"]
    saving = untied_total - expected
    print(cfg)
    print(f"  actual={actual:,}  analytical={expected:,}")
    print(f"  tying saved {saving:,} params (= V*D), {saving/untied_total:.1%} of the untied total")
    assert actual == expected


In [ ]:
"Question: For which of the three configs above does tying save the largest fraction of total params?"
"Answer: "
"Question: As D grows (holding V fixed), does the relative tying saving grow or shrink? Why?"
"Answer: "


In [ ]:
"Question: In the configs above, when do embeddings dominate and when do blocks dominate?"
"Answer: "
"Question: Why does the FFN usually contain more parameters than attention inside one block?"
"Answer: "